##### Setup

###### 1. Importações

In [1]:
from pathlib import Path
from urllib.request import urlopen

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.interpolate import CubicSpline, make_interp_spline

###### 2. Funções compartilhadas de exibição

In [2]:
def exibir_controles(funcao, controles):
    """Executa o exemplo imediatamente e atualiza o gráfico ao mudar controles."""
    saida = widgets.interactive_output(funcao, controles)
    display(widgets.VBox(list(controles.values())), saida)


def configurar_plano(eixo, titulo):
    eixo.set(xlabel="x", ylabel="y", title=titulo)
    eixo.set_aspect("equal", adjustable="box")
    eixo.grid(True, color="#dddddd", linewidth=0.8)
    eixo.set_axisbelow(True)
    eixo.margins(0.12)


def finalizar_figura(figura, arquivo):
    figura.tight_layout()
    figura.savefig(arquivo, dpi=160, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(figura)


# Compartilhada pelas questões 3 e 6.
def curva_bezier(pontos, amostras=100):
    """Avalia a curva cúbica de Bézier nos quatro pontos Nx2."""
    pontos = np.asarray(pontos, dtype=float)
    if pontos.shape != (4, 2):
        raise ValueError("A curva cúbica precisa de quatro pontos 2D")
    t = np.linspace(0, 1, amostras)[:, None]
    return ((1-t)**3 * pontos[0] + 3*t*(1-t)**2 * pontos[1]
            + 3*t**2*(1-t) * pontos[2] + t**3 * pontos[3])

###### 3. Modelo do Fusca

In [3]:
def plano_da_imagem(pontos):
    pontos = np.asarray(pontos, dtype=float)
    return np.column_stack(((pontos[:, 0] - 35) / 44.25,
                            (175 - pontos[:, 1]) / 44.25))


def circulo_em_segmentos(cx, cy, raio, quantidade=24):
    angulos = np.linspace(0, 2*np.pi, quantidade+1)
    return np.column_stack((cx + raio*np.cos(angulos),
                            cy + raio*np.sin(angulos)))


fusca_imagem = [
    np.array([(37, 127), (44, 113), (55, 99), (76, 91), (101, 83),
              (128, 77), (151, 73), (163, 57), (174, 44), (199, 37),
              (230, 33), (258, 35), (284, 41), (308, 51), (338, 76),
              (359, 95), (376, 117), (389, 139)], dtype=float),
    np.array([(37, 127), (35, 139), (56, 139)], dtype=float),
    np.array([(149, 143), (270, 143)], dtype=float),
    np.array([(355, 143), (389, 143), (389, 139)], dtype=float),
    np.array([(56, 143), (62, 125), (76, 113), (93, 107),
              (110, 110), (128, 122), (140, 139), (149, 143)], dtype=float),
    np.array([(270, 143), (280, 122), (296, 109), (319, 105),
              (338, 112), (350, 128), (355, 143)], dtype=float),
    circulo_em_segmentos(92, 145, 34),
    circulo_em_segmentos(92, 145, 23),
    circulo_em_segmentos(319, 145, 34),
    circulo_em_segmentos(319, 145, 23),
    np.array([(151, 82), (172, 46), (181, 45), (181, 82), (151, 82)], dtype=float),
    np.array([(181, 45), (204, 44), (232, 44), (232, 82),
              (181, 82)], dtype=float),
    np.array([(235, 45), (259, 47), (281, 53), (299, 63),
              (310, 74), (299, 83), (235, 83), (235, 45)], dtype=float),
    np.array([(151, 82), (299, 82)], dtype=float),
    np.array([(232, 44), (232, 151), (270, 151)], dtype=float),
    np.array([(149, 143), (149, 151), (232, 151)], dtype=float),
    np.array([(219, 91), (236, 91)], dtype=float),
    np.array([(54, 97), (79, 91), (104, 85), (150, 78)], dtype=float),
    np.array([(338, 76), (340, 79), (310, 79)], dtype=float),
    np.array([(35, 127), (32, 128)], dtype=float),
    np.array([(389, 139), (390, 146), (375, 146)], dtype=float),
]
fusca = [plano_da_imagem(pontos) for pontos in fusca_imagem]

# Referências para os trechos usados nas questões seguintes.
modelo_fusca = {
    "contorno": fusca[0],
    "arco_dianteiro": fusca[4],
    "arco_traseiro": fusca[5],
    "teto": fusca[0][6:15],
}

# Métodos compartilhados pelas questões 2 e 6.
def spline_do_contorno(trecho, pontos_extras=0, amostras=200):
    x, y = np.asarray(trecho, dtype=float).T
    if pontos_extras:
        lacunas = np.argsort(np.diff(x))[-min(pontos_extras, len(x)-1):]
        novos_x = (x[lacunas] + x[lacunas + 1]) / 2
        x_controle = np.sort(np.concatenate((x, novos_x)))
        y_controle = np.interp(x_controle, x, y)
    else:
        x_controle, y_controle = x, y
    amostras_x = np.linspace(x_controle[0], x_controle[-1], amostras)
    curva = CubicSpline(x_controle, y_controle, bc_type="natural")
    return (np.column_stack((x_controle, y_controle)),
            np.column_stack((amostras_x, curva(amostras_x))))


# Métodos compartilhados pelas questões 3 e 6.
def controles_bezier_arco(arco, curvatura=1):
    inicio, fim = arco[0], arco[-1]
    altura_arco = np.max(arco[:, 1])
    distancia = fim[0] - inicio[0]
    return np.array([
        inicio,
        [inicio[0] + distancia/3,
         inicio[1] + (altura_arco - inicio[1])*4/3*curvatura],
        [fim[0] - distancia/3,
         fim[1] + (altura_arco - fim[1])*4/3*curvatura],
        fim,
    ])


# Métodos compartilhados pelas questões 4 e 6.
def bspline_do_teto(teto, altura=1, pontos_extras=0, amostras=250):
    x_base, y_base = np.asarray(teto, dtype=float).T
    linha_base = np.interp(x_base, [x_base[0], x_base[-1]],
                           [y_base[0], y_base[-1]])
    y_altura = linha_base + (y_base - linha_base)*altura
    x_extra = ((x_base[:-1] + x_base[1:]) / 2)[:pontos_extras]
    x_controle = np.sort(np.concatenate((x_base, x_extra)))
    y_controle = np.interp(x_controle, x_base, y_altura)
    amostras_x = np.linspace(x_controle[0], x_controle[-1], amostras)
    spline = make_interp_spline(x_controle, y_controle, k=3)
    return (np.column_stack((x_controle, y_controle)),
            np.column_stack((amostras_x, spline(amostras_x))))

def configurar_vista_fusca(eixo, escala_x=1, escala_y=1, limite_superior=None):
    eixo.set_aspect("equal", adjustable="box")
    eixo.set_xlim(-0.35*escala_x, 8.3*escala_x)
    topo = 3.5*escala_y
    if limite_superior is not None:
        topo = max(topo, limite_superior)
    eixo.set_ylim(-0.3*escala_y, topo)
    eixo.axis("off")


##### QUESTÃO1

###### 1.

In [4]:
formas = {
    "Fusca": fusca,
    "Casa": [np.array([(0, 0), (0, 3), (2, 5), (4, 3),
                        (4, 0), (0, 0)], dtype=float)],
    "Estrela": [np.array([(0, 3), (1, 0), (4, 0), (1.6, -1.8),
                          (2.6, -4.5), (0, -2.8), (-2.6, -4.5),
                          (-1.6, -1.8), (-4, 0), (-1, 0), (0, 3)], dtype=float)],
    "Coração": [np.array([(0, -3), (-1.5, -1.5), (-3, 0.5), (-3, 2),
                           (-2, 3), (-1, 3), (0, 2), (1, 3), (2, 3),
                           (3, 2), (3, 0.5), (1.5, -1.5), (0, -3)], dtype=float)],
}


def mostrar_polilinha(forma, escala):
    figura, eixo = plt.subplots(figsize=(10, 4.5))
    for pontos in formas[forma]:
        vertices = pontos * escala
        eixo.plot(vertices[:, 0], vertices[:, 1], "-o",
                  color="#dc8a1d" if forma == "Fusca" else "tab:blue",
                  linewidth=1.3 if forma == "Fusca" else 2,
                  markersize=1.8 if forma == "Fusca" else 5)
    if forma == "Fusca":
        configurar_vista_fusca(eixo, escala_x=escala, escala_y=escala)
    else:
        configurar_plano(eixo, f"Questão 1 — Polilinha: {forma}")
    finalizar_figura(figura, "lista4_questao1_polilinha.png")


exibir_controles(mostrar_polilinha, {
    "forma": widgets.Dropdown(options=list(formas), value="Fusca", description="Forma"),
    "escala": widgets.FloatSlider(value=1, min=0.5, max=2, step=0.1,
                                  description="Escala", continuous_update=False),
})

Output()

##### QUESTÃO2

###### 1

In [5]:
def mostrar_spline_cubica(pontos_extras, altura):
    partes = [pontos * np.array([1.0, altura]) for pontos in fusca]
    figura, eixos = plt.subplots(1, 2, figsize=(14, 5))

    for indice, eixo in enumerate(eixos):
        for parte, vertices in enumerate(partes):
            x, y = vertices.T
            if indice == 1 and parte in (0, 4, 5):
                # O capô, o teto e a traseira encontram-se em quinas reais.
                # Ajustá-los separadamente evita uma ondulação nessa junção.
                trechos = ([vertices[:7], vertices[6:15], vertices[14:]]
                           if parte == 0 else [vertices])
                for trecho in trechos:
                    controles, curva = spline_do_contorno(trecho, pontos_extras)
                    eixo.plot(curva[:, 0], curva[:, 1], color="#dc8a1d", linewidth=1.5)
                    eixo.scatter(controles[:, 0], controles[:, 1],
                                 color="#dc8a1d", s=8)
            else:
                eixo.plot(x, y, "-o", color="#888888" if indice == 0 else "#dc8a1d",
                          linewidth=1.2, markersize=1.8)
        configurar_vista_fusca(eixo, escala_y=altura)
        eixo.set_title("Polilinhas da Questão 1" if indice == 0
                       else "Contorno suavizado com spline cúbica")

    finalizar_figura(figura, "lista4_questao2_spline.png")


exibir_controles(mostrar_spline_cubica, {
    "pontos_extras": widgets.IntSlider(value=0, min=0, max=4, step=1,
                                        description="Extras/trecho", continuous_update=False),
    "altura": widgets.FloatSlider(value=1, min=0.5, max=2, step=0.1,
                                   description="Altura", continuous_update=False),
})

Output()

##### QUESTÃO3 

###### 1

In [6]:
def mostrar_bezier(y1, y2):
    figura, eixo = plt.subplots(figsize=(10, 4.5))
    arcos = {"arco_dianteiro": y1, "arco_traseiro": y2}

    # Todo o Fusca vem do mesmo modelo; os arcos originais ficam como referência.
    for parte, pontos in enumerate(fusca):
        eixo.plot(pontos[:, 0], pontos[:, 1], "o", color="#b0b0b0",
                  linewidth=1, markersize=1.5,
                  linestyle="--" if parte in (4, 5) else "-")

    for nome, curvatura in arcos.items():
        arco = modelo_fusca[nome]
        pontos_controle = controles_bezier_arco(arco, curvatura)
        bezier = curva_bezier(pontos_controle, amostras=150)
        eixo.plot(pontos_controle[:, 0], pontos_controle[:, 1], "o--",
                  color="#b75e2c", linewidth=0.9, markersize=4)
        eixo.plot(bezier[:, 0], bezier[:, 1], color="#df8420", linewidth=2.4)

    configurar_vista_fusca(eixo)
    eixo.set_title("Questão 3 — Bézier nos para-lamas do Fusca")
    finalizar_figura(figura, "lista4_questao3_bezier.png")


exibir_controles(mostrar_bezier, {
    "y1": widgets.FloatSlider(value=1, min=0.4, max=1.6, step=0.05,
                              description="Dianteiro", continuous_update=False),
    "y2": widgets.FloatSlider(value=1, min=0.4, max=1.6, step=0.05,
                              description="Traseiro", continuous_update=False),
})

Output()

##### QUESTÃO4 

In [7]:
def mostrar_bspline(pontos_extras, altura):
    teto = modelo_fusca["teto"]
    controles, curva = bspline_do_teto(teto, altura, pontos_extras)
    x_base, y_base = teto.T

    figura, eixo = plt.subplots(figsize=(10, 4.5))
    for pontos in fusca[1:]:
        eixo.plot(pontos[:, 0], pontos[:, 1], color="#b0b0b0", linewidth=1)
    contorno = modelo_fusca["contorno"]
    # Capô e traseira permanecem no desenho; só o trecho do teto muda.
    for trecho in (contorno[:7], contorno[14:]):
        eixo.plot(trecho[:, 0], trecho[:, 1], color="#b0b0b0", linewidth=1)
    eixo.plot(x_base, y_base, "o--", color="#aaaaaa", markersize=3,
              label="Teto original")
    eixo.plot(controles[:, 0], controles[:, 1], "o", color="#c45f1c",
              markersize=4, label="Pontos de controle")
    eixo.plot(curva[:, 0], curva[:, 1], color="#df8420", linewidth=2.5,
              label="B-Spline cúbica")
    configurar_vista_fusca(eixo, limite_superior=np.max(curva[:, 1]) + 0.3)
    eixo.set_title("Questão 4 — B-Spline no teto do Fusca")
    eixo.legend(loc="upper right", fontsize=8)
    finalizar_figura(figura, "lista4_questao4_bspline.png")


exibir_controles(mostrar_bspline, {
    "pontos_extras": widgets.IntSlider(value=0, min=0, max=8, step=1,
                                        description="Pontos extras", continuous_update=False),
    "altura": widgets.FloatSlider(value=1, min=0.5, max=2, step=0.1,
                                   description="Altura", continuous_update=False),
})

Output()

##### QUESTÃO5

In [8]:
def carregar_suzanne():
    caminho = Path("Archives/suzanne.obj")
    if not caminho.is_file():
        caminho = Path("suzanne.obj")
    if not caminho.is_file():
        fonte = ("https://raw.githubusercontent.com/alecjacobson/"
                 "common-3d-test-models/master/data/suzanne.obj")
        with urlopen(fonte, timeout=30) as resposta:
            caminho.write_bytes(resposta.read())
    vertices, faces = [], []
    with caminho.open(encoding="utf-8") as arquivo:
        for linha in arquivo:
            if linha.startswith("v "):
                vertices.append([float(valor) for valor in linha.split()[1:4]])
            elif linha.startswith("f "):
                faces.append([int(parte.split("/")[0]) - 1
                              for parte in linha.split()[1:]])
    vertices = np.asarray(vertices, dtype=float)
    if not len(vertices) or not len(faces):
        raise ValueError("OBJ sem vértices ou faces")
    vertices -= (vertices.min(axis=0) + vertices.max(axis=0)) / 2
    return vertices, faces


vertices_suzanne, faces_suzanne = carregar_suzanne()


def malha_grade(n, superficie):
    """Gera vértices e faces quadradas ou triangulares de uma grade."""
    u = np.linspace(-1, 1, n)
    x, y = np.meshgrid(u, u)
    z = 0.32 * np.cos(np.pi*x) * np.cos(np.pi*y)
    vertices = np.column_stack((x.ravel(), y.ravel(), z.ravel()))
    faces = []
    for i in range(n-1):
        for j in range(n-1):
            a = i*n + j
            quad = [a, a+1, a+n+1, a+n]
            if superficie == "triangular":
                faces.extend(([quad[0], quad[1], quad[2]],
                              [quad[0], quad[2], quad[3]]))
            else:
                faces.append(quad)
    return vertices, faces


def malha_esfera(n):
    latitudes = np.linspace(0, np.pi, n+1)
    longitudes = np.linspace(0, 2*np.pi, 2*n+1)[:-1]
    vertices = np.array([[np.sin(t)*np.cos(p), np.sin(t)*np.sin(p), np.cos(t)]
                         for t in latitudes for p in longitudes])
    m = len(longitudes)
    faces = []
    for i in range(n):
        for j in range(m):
            a = i*m+j
            b = i*m+(j+1)%m
            c = (i+1)*m+(j+1)%m
            d = (i+1)*m+j
            if i > 0:
                faces.append([a, b, d])
            if i < n-1:
                faces.append([b, c, d])
    return vertices, faces


def mostrar_malhas(densidade, elevacao, azimute):
    malhas = [
        (*malha_esfera(densidade), "(a) Esfera poligonal"),
        (vertices_suzanne, faces_suzanne, "(b) Suzanne"),
        (*malha_grade(densidade, "quadrilateral"), "(c) Quadriláteros"),
        (*malha_grade(densidade, "triangular"), "(d) Triângulos"),
    ]
    figura = plt.figure(figsize=(12, 10))
    for indice, (vertices, faces, titulo) in enumerate(malhas, 1):
        eixo = figura.add_subplot(2, 2, indice, projection="3d")
        malha = Poly3DCollection([vertices[face] for face in faces],
                                 facecolor="#7db7d4", edgecolor="#34495e",
                                 linewidth=0.3, alpha=0.8)
        eixo.add_collection3d(malha)
        if indice != 2:
            eixo.scatter(vertices[:, 0], vertices[:, 1], vertices[:, 2],
                         color="tab:red", s=4)
        limite = np.max(np.abs(vertices)) * 1.15
        eixo.set(xlim=(-limite, limite), ylim=(-limite, limite),
                 zlim=(-limite, limite), title=titulo)
        eixo.set_box_aspect((1, 1, 1))
        eixo.view_init(elev=elevacao, azim=azimute)
    finalizar_figura(figura, "lista4_questao5_malhas.png")


exibir_controles(mostrar_malhas, {
    "densidade": widgets.IntSlider(value=5, min=3, max=12, step=1,
                                    description="Densidade", continuous_update=False),
    "elevacao": widgets.IntSlider(value=25, min=-30, max=80, step=5,
                                   description="Elevação", continuous_update=False),
    "azimute": widgets.IntSlider(value=-60, min=-180, max=180, step=5,
                                  description="Azimute", continuous_update=False),
})

Output()

##### QUESTÃO6

In [9]:
def mostrar_fusca(altura_teto, comprimento, curvatura_lamas):
    figura, eixo = plt.subplots(figsize=(11, 5))

    def traco(pontos, **estilo):
        eixo.plot(pontos[:, 0] * comprimento, pontos[:, 1], **estilo)

    # A carroceria, as rodas, a porta e as janelas vêm do modelo do setup.
    for indice, pontos in enumerate(fusca[1:], start=1):
        if indice not in (4, 5):
            traco(pontos, color="#59656d", linewidth=1.2,
                  label="Polilinhas: corpo e detalhes" if indice == 1 else None)

    # Splines cúbicas seguem o capô e a traseira do mesmo contorno.
    contorno = modelo_fusca["contorno"]
    for indice, trecho in enumerate((contorno[:7], contorno[14:])):
        _, curva = spline_do_contorno(trecho)
        traco(curva, color="#3484a6", linewidth=2.3,
              label="Spline: capô e traseira" if indice == 0 else None)

    # A B-Spline altera somente o teto e preserva suas duas extremidades.
    _, teto = bspline_do_teto(modelo_fusca["teto"], altura_teto)
    traco(teto, color="#d87636", linewidth=2.8, label="B-Spline: teto")

    # Bézier substitui os dois para-lamas e acompanha o alto da janela traseira.
    for indice, nome in enumerate(("arco_dianteiro", "arco_traseiro")):
        controles = controles_bezier_arco(modelo_fusca[nome], curvatura_lamas)
        traco(curva_bezier(controles), color="#9b59b6", linewidth=2.3,
              label="Bézier: para-lamas" if indice == 0 else None)
    janela_traseira = fusca[12]
    controles_janela = janela_traseira[[0, 1, 3, 4]]
    traco(curva_bezier(controles_janela), color="#9b59b6", linewidth=1.8,
          label="Bézier: janela")

    configurar_vista_fusca(eixo, escala_x=comprimento,
                           limite_superior=np.max(teto[:, 1]) + 0.3)
    eixo.set_title("Questão 6 — Curvas do Fusca")
    eixo.legend(loc="upper center", bbox_to_anchor=(0.5, -0.04),
               fontsize=8, ncol=3)
    finalizar_figura(figura, "lista4_questao6_fusca.png")


exibir_controles(mostrar_fusca, {
    "altura_teto": widgets.FloatSlider(value=1, min=0.7, max=1.3, step=0.05,
                                        description="Altura teto", continuous_update=False),
    "comprimento": widgets.FloatSlider(value=1, min=0.8, max=1.2, step=0.05,
                                        description="Comprimento", continuous_update=False),
    "curvatura_lamas": widgets.FloatSlider(value=1, min=0.5, max=1.5, step=0.05,
                                            description="Para-lamas", continuous_update=False),
})

Output()